<center>
<a href="https://www.umontpellier.fr/"><img src="https://www.umontpellier.fr/wp-content/uploads/2022/10/logo_um_2022_rouge_rvb.svg" width="200"/></a>&nbsp;&nbsp;
<a href="https://economie.edu.umontpellier.fr/"><img src="https://economie.edu.umontpellier.fr/files/2014/12/economie_rvb_2015-300x137.png" width="160"/></a>
</center>

<div align="center">

#  Phase 3 — Nettoyage et Validation de la Base `temps.csv`

| Nom et Prénom | Rôle |
|---|---|
| Randriamisaina Tsiory-Fanomezana | Membre de l'équipe |
| SHIRALI POUR Amir | Membre de l'équipe |

</div>

---
## Objectif de cette phase

La base `temps.csv` est le **journal des interventions** : chaque ligne enregistre
un acte de traitement d'un dossier par un agent. Un même dossier peut apparaître
plusieurs fois (plusieurs agents, plusieurs sessions de travail).

Ce notebook applique les corrections décrites dans [`docs/phase3_temps.md`](../docs/phase3_temps.md).

**Statistiques initiales connues :**
- 431 598 lignes × 5 colonnes
- `duree.corrigee` : **47 471 NaN** (11%), max = 253 014 min (≈4 215h)
- Dates déjà en format ISO `YYYY/MM/DD`


---
## Section 0 — Initialisation

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

def localiser_racine_du_projet():
    """Remonte l'arborescence jusqu'à trouver la racine (présence de .git ou requirements.txt)."""
    repertoire_courant = Path.cwd()
    while True:
        if any((repertoire_courant / m).exists() for m in ['.git', 'requirements.txt']):
            break
        if repertoire_courant.parent == repertoire_courant:
            break
        repertoire_courant = repertoire_courant.parent
    if Path.cwd() != repertoire_courant:
        os.chdir(repertoire_courant)
    return repertoire_courant.resolve()

REPERTOIRE_RACINE = localiser_racine_du_projet()
sys.path.insert(0, str(REPERTOIRE_RACINE / 'src'))
from utils.dataframe_styler import style_duplicates
print(f" Racine du projet : {REPERTOIRE_RACINE}")

In [ ]:
# --- Chargement des bases
temps_original  = pd.read_csv('data/temps.csv', encoding='latin-1')
dossier_nettoye = pd.read_csv('data/dossier_nettoye.csv', encoding='utf-8')
ressources_nettoyees = pd.read_csv('data/ressources_nettoyees.csv', encoding='utf-8')

# Copie de travail
temps = temps_original.copy()

print(f" temps          : {temps.shape[0]:,} lignes × {temps.shape[1]} colonnes")
print(f" dossier_nettoye: {dossier_nettoye.shape[0]:,} lignes")
print(f" ressources     : {ressources_nettoyees.shape[0]:,} lignes")
print()
print("Types de données :")
print(temps.dtypes)

---
## Section 1 — Vue d'ensemble de la qualité des données

In [ ]:
def compter_anomalies_par_colonne(dataframe):
    """Calcule NaN, ???, total et % pour chaque colonne. Retourne DataFrame trié."""
    resultats = []
    for colonne in dataframe.columns:
        n_nan           = dataframe[colonne].isna().sum()
        n_interrogation = (dataframe[colonne].astype(str).str.strip() == '???').sum()
        total           = n_nan + n_interrogation
        resultats.append({
            'Colonne'        : colonne,
            'NaN'            : n_nan,
            '???'            : n_interrogation,
            'Total anomalies': total,
            '% du total'     : round(total / len(dataframe) * 100, 2)
        })
    return (pd.DataFrame(resultats)
              .sort_values('Total anomalies', ascending=False)
              .reset_index(drop=True))

tableau_anomalies_initial = compter_anomalies_par_colonne(temps)
print("Tableau récapitulatif des anomalies initiales :")
tableau_anomalies_initial

---
## Section 2 — Anomalie C1 : NaN dans `duree.corrigee` (47 471 lignes)

### Constat
47 471 lignes (11%) n'ont pas de durée de traitement enregistrée.

### Décision : Conservation avec NaN — aucune imputation
> Imputer par la médiane introduirait un biais dans les analyses de performance des agents.  
> Ces lignes restent utiles pour les analyses de fréquence et de présence.  
> Elles seront **exclues uniquement** des modèles nécessitant cette variable.

In [ ]:
n_nan_duree = temps['duree.corrigee'].isna().sum()
pct_nan     = n_nan_duree / len(temps) * 100

print(f"NaN dans duree.corrigee : {n_nan_duree:,} ({pct_nan:.2f}%)")
print()
print("Statistiques des valeurs non-NaN :")
print(temps['duree.corrigee'].describe())
print()
print("Aperçu d'exemples avec NaN :")
temps[temps['duree.corrigee'].isna()].head(5).style_duplicates(
    highlight_mask=temps['duree.corrigee'].isna()
)

---
## Section 3 — Anomalie C2 : Valeurs aberrantes dans `duree.corrigee`

### Constat
Maximum observé : **253 014 minutes** (≈ 175 jours). La médiane est 188 min.

### Décision : Signalement avec seuil de vigilance — conservation sans suppression
> Seuil d'alerte : durée > **1 440 min** (24 heures = 1 journée complète).  
> Ces cas seront traités comme outliers dans les modèles (Phase 5 & 6).